# ForgeEdge — Rule Discovery (Modulo 3)

Questo notebook testa il modulo **RuleDiscovery**, il quarto passo della pipeline FORGE, partendo da un dato letto da un file **Excel in locale**.

La catena è strettamente sequenziale:

```
Market Context  →  Event Discovery  →  Alpha Discovery  →  Rule Discovery
```

Alpha Discovery produce gli **Alpha Contract** (pattern con evidenza statistica e target derivato). Rule Discovery riceve ogni contratto **e il suo Event Candidate**, ricostruisce il segnale dell'evento con i parametri salvati sul candidato (stessa attivazione bit-per-bit di Event/Alpha Discovery), parametrizza la meccanica di un ordine limite realistico (fee, fill rate, target discreto) e risponde alla domanda operativa: **EDGE / PARTIAL-EDGE / NON-EDGE**.

Il notebook copre:
1. backtest di una regola base e confronto con le metriche attese;
2. screening della griglia dei parametri operativi;
3. validazione statistica (t-test, Deflated Sharpe, stabilità temporale);
4. validazione **out-of-sample walk-forward** + envelope/MAE-MFE IS e OOS;
5. confronto **long vs short** sulla stessa regola.


## 0. Setup


In [ ]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from forgedge import (
    MarketContext,
    EventDiscovery, DiscoveryConfig,
    AlphaDiscovery, AlphaConfig,
    RuleDiscovery, RuleDiscoveryConfig,
)
from forgedge.rule_discovery import (
    BacktestParams, GridSpec, ScoringParams, SelectionCriteria, WalkForwardConfig,
    run_backtest, text_report, html_report,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


## 1. Caricamento da Excel in locale

Il dataset di esempio contiene candele orarie per uno o più simboli (es. ADAUSDC, DOGEUSDC). Filtriamo su un singolo simbolo, ordiniamo cronologicamente e costruiamo la colonna timestamp `open_dt`.

> Sostituisci `DATA_PATH` con il percorso al tuo file.


In [ ]:
DATA_PATH = "../data/test1h.xlsx"
SYMBOL    = "ADAUSDC"

df_raw = pd.read_excel(DATA_PATH)
df = df_raw[df_raw["symbol"] == SYMBOL].copy().sort_values("open_time").reset_index(drop=True)
df["open_dt"] = pd.to_datetime(df["open_time"], unit="ms")
df = df.drop(columns=[c for c in ("symbol", "timeframe", "open_time") if c in df.columns])
df = df.dropna().reset_index(drop=True)

print(f"Simbolo : {SYMBOL}")
print(f"Righe   : {len(df):,}")
print(f"Range   : {df['open_dt'].iloc[0].date()} → {df['open_dt'].iloc[-1].date()}")
df.head(3)


## 2. Modulo 0 — Market Context

`MarketContext` etichetta ogni barra con un `regime` (e `regime_stable`). Rule Discovery lo legge per il breakdown per regime — non lo ricalcola.


In [ ]:
mc = MarketContext(df.copy())
enriched = mc.run()
print("Distribuzione dei regimi:")
print(mc.distribution())


## 3. Modulo 1 — Event Discovery

`EventDiscovery` genera gli Event Candidate (condizioni booleane sulle feature) **senza guardare il forward return**. La tabella post-pipeline `ed.df` contiene `regime` e tutte le feature derivate: è quella che passeremo sia ad Alpha che a Rule Discovery.


In [ ]:
ed = EventDiscovery(enriched.copy(), DiscoveryConfig(timestamp_col="open_dt"))
candidates = ed.run()
print(f"Event Candidates: {len(candidates)}")


## 4. Modulo 2 — Alpha Discovery

`AlphaDiscovery` deriva dai dati il target economico di ogni evento (orizzonte, sell_pct, direzione), lo conferma out-of-sample e promuove i candidati con evidenza statistica → **Alpha Contract**.


In [ ]:
ad = AlphaDiscovery(ed.df, candidates, AlphaConfig(asset=SYMBOL, timeframe="1H"))
ad.run()
promoted = ad.promoted_contracts()
by_id = {c.event_id: c for c in candidates}
print(f"Contratti valutati  : {len(ad._contracts)}")
print(f"Promossi (HYPOTHESIS): {len(promoted)}")

cols = ["expression", "direction", "holding_period_h", "sell_pct", "win_rate", "lift", "grade"]
ad.summary()[ad.summary()["promoted"]].head(8)[cols]


## 5. Modulo 3 — Rule Discovery: configurazione

La configurazione esplora **solo i parametri operativi** (la regola resta fissa). Il target del contratto (`sell_pct`, `target_h`, `direction`) viene usato come centro del grid.

| Sezione | Parametro | Significato |
|---|---|---|
| `base_params` | `buy_type`, `buy_drop_pct`, `fee`, `early_stopping` | meccanica fissa dell'ordine |
| `grid` | `buy_drop_pct`, `sell_pct`, `target_h` | assi esplorati in-sample |
| `scoring` | `pf_tpm_target` | target trade/mese per `pf_score_tpm` |
| `walk_forward` | `n_splits`, `min_train_months` | validazione OOS walk-forward |
| `criteria` | `min_profit_factor`, `min_win_rate` | soglie del verdetto |


In [ ]:
cfg = RuleDiscoveryConfig(
    base_params=BacktestParams(
        buy_type="limit", buy_drop_pct=0.010, buy_delay_bar=6,
        fee=0.002, early_stopping=True,
    ),
    grid=GridSpec(
        buy_drop_pct=[0.006, 0.010, 0.014],
        sell_pct=[0.030, 0.040, 0.050],
        target_h=[12, 24, 48],
    ),
    scoring=ScoringParams(pf_min_trades=15, pf_min_tpm=2, pf_tpm_target=3),
    walk_forward=WalkForwardConfig(n_splits=4, min_train_months=6),
    criteria=SelectionCriteria(min_profit_factor=2.0, min_win_rate=0.55),
)


## 6. Esecuzione su un contratto e verdetto

Scegliamo il contratto col composite score più alto e lanciamo `RuleDiscovery`. Il `text_report` riassume verdetto, in-sample, validazione statistica, walk-forward OOS, envelope, MAE/MFE e regime.


In [ ]:
contract = promoted[0]
cand = by_id[contract.event_candidate_id]

rd = RuleDiscovery(ed.df, contract, cand, cfg)
resp = rd.run()

print(text_report(resp))


## 7. Screening della griglia in-sample

Una riga per combinazione, ordinata per `pf_score_tpm` (PF bilanciato dalla consistenza mensile). È il criterio con cui viene scelto il punto operativo, per evitare l'overfitting sul PF grezzo.


In [ ]:
grid_df = rd.grid_summary()
print(f"Combinazioni valutate: {len(grid_df)}")
grid_df.head(10)


## 8. Walk-forward out-of-sample

Per ogni split i parametri vengono **ri-selezionati sul train** e valutati una sola volta sul **test** mai visto. La concatenazione dei trade di test è il track record OOS onesto.


In [ ]:
wf = resp.walk_forward
if wf is None:
    print("Walk-forward non disponibile (storia troppo corta per uno split).")
else:
    rows = []
    for s in wf.splits:
        ts = s.test_summary
        rows.append({
            "split": s.split_idx,
            "train": f"{s.train_from} → {s.train_to}",
            "test":  f"{s.test_from} → {s.test_to}",
            "PF":     round(ts.profit_factor, 3),
            "WR":     round(ts.win_rate_pct, 3),
            "trades": ts.total_trades,
            "net":    round(ts.total_net_gain, 4),
        })
    wf_df = pd.DataFrame(rows)
    display(wf_df)
    print(f"\nOOS aggregato: PF={wf.oos_summary.profit_factor:.3f}  "
          f"WR={wf.oos_summary.win_rate_pct:.3f}  "
          f"trades={wf.oos_summary.total_trades}  "
          f"consistency={wf.consistency:.0%}")


In [ ]:
# Profit factor per split (linea di break-even a PF=1)
if wf is not None:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    pf = [s.test_summary.profit_factor for s in wf.splits]
    colors = ["#2ecc71" if p >= 1 else "#e74c3c" for p in pf]
    ax.bar([s.split_idx for s in wf.splits], pf, color=colors, alpha=0.8)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, label="break-even (PF=1)")
    ax.set_title("Walk-forward — Profit Factor per finestra di test (OOS)")
    ax.set_xlabel("split"); ax.set_ylabel("PF"); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()


## 9. Range d'azione — execution envelope e MAE/MFE (IS e OOS)

Senza scegliere una singola convenzione di uscita, il modulo riporta la performance tra `close` (conservativa, = motore certificato) e l'estremo intrabar ottimistico (`high` per long, `low` per short), più l'escursione MAE/MFE intra-trade. Tutte e due le diagnostiche sono calcolate sia **in-sample** sia **out-of-sample** (sui trade walk-forward).


In [ ]:
def envelope_row(env, label):
    c, o = env.conservative, env.optimistic
    return {
        "scope": label,
        "PF_cons (close)": round(c.profit_factor, 3),
        "PF_opt (intrabar)": round(o.profit_factor, 3),
        "exp_cons": round(c.expectancy, 5),
        "exp_opt": round(o.expectancy, 5),
        "hit_cons%": round(c.target_hit_rate_pct, 1),
        "hit_opt%": round(o.target_hit_rate_pct, 1),
    }

env_rows = [envelope_row(resp.execution_envelope, "in-sample")]
if wf is not None and wf.oos_envelope is not None:
    env_rows.append(envelope_row(wf.oos_envelope, "out-of-sample"))
display(pd.DataFrame(env_rows))

def mae_mfe_row(ex, label):
    return {"scope": label,
            "MAE_mean": round(ex.mae_mean, 4), "MAE_worst": round(ex.mae_worst, 4),
            "MFE_mean": round(ex.mfe_mean, 4), "MFE_best": round(ex.mfe_best, 4),
            "MFE→target%": round(ex.mfe_reached_target_pct, 1)}

mm_rows = []
if resp.excursion is not None:
    mm_rows.append(mae_mfe_row(resp.excursion, "in-sample"))
if wf is not None and wf.oos_excursion is not None:
    mm_rows.append(mae_mfe_row(wf.oos_excursion, "out-of-sample"))
pd.DataFrame(mm_rows)


## 10. Report HTML autocontenuto

Lo stesso verdetto come HTML inline (tabelle, nessuna dipendenza esterna). Utile da salvare o incorporare.


In [ ]:
display(HTML(html_report(resp)))

# Per salvarlo su file:
# with open("rule_discovery_report.html", "w") as f:
#     f.write(html_report(resp))


## 11. Backtest diretto di una regola base

Il motore `run_backtest` è usabile anche standalone: basta una tabella candele con `open_dt` e una colonna-segnale booleana. Qui testiamo la regola base **`close_rsi_14 < 30`** (oversold, bias mean-reversion long).

Il rendimento è ancorato al **prezzo di fill** (non al close del segnale): take-profit `fill·(1+sell_pct)`, net `(exit−fill)/fill` al netto delle fee round-trip.


In [ ]:
bt = df.copy()
bt["rsi30"] = (bt["close_rsi_14"] < 30).astype(int)

params = BacktestParams(
    direction="long", buy_type="limit", buy_drop_pct=0.02, buy_delay_bar=4,
    sell_pct=0.06, target_h=24, fee=0.002, early_stopping=True,
)
summary, trades = run_backtest(bt, "rsi30", params, return_trades=True)

print(f"segnali       : {summary.total_signals}")
print(f"trade         : {summary.total_trades}  (fill rate {summary.fill_rate:.1%})")
print(f"win rate      : {summary.win_rate_pct:.1%}")
print(f"profit factor : {summary.profit_factor:.3f}")
print(f"expectancy    : {summary.expectancy:+.4%}")
print(f"net gain tot. : {summary.total_net_gain:+.2%}")
print(f"pf_score_tpm  : {summary.pf_score_tpm:.4f}   tpm_mu={summary.tpm_mu:.1f}")


In [ ]:
# Equity curve (somma cumulata del net per trade)
t = trades.sort_values("fill_dt").reset_index(drop=True)
t["cum"] = t["net_pct_gain"].cumsum()
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(t["fill_dt"], t["cum"], color="steelblue", linewidth=1.3)
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.fill_between(t["fill_dt"], t["cum"], 0, where=t["cum"] >= 0, alpha=0.15, color="green")
ax.fill_between(t["fill_dt"], t["cum"], 0, where=t["cum"] < 0, alpha=0.20, color="red")
ax.set_title(f"{SYMBOL} — close_rsi_14 < 30 (long) — P&L cumulato per trade")
ax.set_ylabel("net gain cumulato"); plt.tight_layout(); plt.show()


## 12. Long vs Short sulla stessa regola

`direction` è un parametro libero applicabile a qualsiasi regola. Proprietà attesa:
- su una regola con **edge direzionale** (es. `close_rsi_14 < 30`, mean-reversion long) le due direzioni **divergono** — lo short è l'inverso;
- su una regola **non-edge** (segnale casuale) le due direzioni si comportano in modo simile (fill rate speculari, entrambe in perdita).

Il fill è speculare: il long controlla il `low` delle prossime N candele (`low ≤ entry`), lo short controlla l'`high` (`high ≥ entry`).


In [ ]:
def run_dir(signal_col, direction):
    p = BacktestParams(direction=direction, buy_type="limit", buy_drop_pct=0.02,
                       buy_delay_bar=4, sell_pct=0.06, target_h=24, fee=0.002)
    s = run_backtest(bt, signal_col, p)
    return {"direction": direction, "trades": s.total_trades,
            "fill_rate": round(s.fill_rate, 3), "PF": round(s.profit_factor, 3),
            "win_rate": round(s.win_rate_pct, 3), "expectancy": round(s.expectancy, 5)}

# Regola EDGE
rng = np.random.default_rng(0)
bt["rand"] = (rng.random(len(bt)) < float(bt["rsi30"].mean())).astype(int)

print("EDGE  — close_rsi_14 < 30")
display(pd.DataFrame([run_dir("rsi30", "long"), run_dir("rsi30", "short")]))

print("NON-EDGE — segnale casuale (stesso tasso di attivazione)")
display(pd.DataFrame([run_dir("rand", "long"), run_dir("rand", "short")]))


## 13. Batch — verdetti su tutti i contratti promossi

Eseguiamo Rule Discovery su tutti i contratti promossi e contiamo i verdetti. Le regole con edge OOS sono quelle che superano il walk-forward.


In [ ]:
from collections import Counter

verdicts = Counter()
edges = []
for c in promoted:
    r = RuleDiscovery(ed.df, c, by_id[c.event_candidate_id], cfg).run()
    verdicts[r.verdict] += 1
    if r.is_edge:
        edges.append(r)

print("Verdetti:")
for v in ("EDGE", "PARTIAL-EDGE", "NON-EDGE"):
    print(f"  {v:<13}: {verdicts.get(v, 0)}")

# Migliori per PF out-of-sample
def oos_pf(r):
    return r.walk_forward.oos_summary.profit_factor if r.walk_forward else float("nan")

rows = [{
    "alpha_id": r.alpha_id,
    "verdict": r.verdict,
    "expression": (r.validated_rule.expression if r.validated_rule else "")[:60],
    "IS_PF": round(r.in_sample_summary.profit_factor, 3),
    "OOS_PF": round(oos_pf(r), 3),
} for r in sorted(edges, key=oos_pf, reverse=True)]
pd.DataFrame(rows).head(10)


---

*Rule Discovery Pipeline — FORGE (Feature-Oriented Rule Generation Engine)*  
*Documento tecnico di ricerca. Non costituisce consulenza finanziaria.*
